In [ ]:
# 05_additional_experiments.ipynb
# Additional experiments (CSDA): finite-sample coverage of the normal
# approximation, Kolmogorov distance to normality, robustness to anisotropic
# orientation, and computational cost. Raise `reps` toward the shipped values
# (8000 for n<=400, 3500 for n=800) for publication-quality numbers.
import os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"font.size": 10, "axes.edgecolor": "0.3"})
GRAY = "0.35"
FIG_DIR, DATA_DIR = "../results/figures", "../data"
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

rng = np.random.default_rng(20261012)
ell = np.array([0.2, 0.4]); p = np.array([0.5, 0.5]); twor = 0.02


def seg_seg_dist(P1, P2, Q1, Q2):
    d1, d2, r = P2 - P1, Q2 - Q1, P1 - Q1
    a = (d1 * d1).sum(1); e = (d2 * d2).sum(1); f = (d2 * r).sum(1)
    b = (d1 * d2).sum(1); c = (d1 * r).sum(1)
    den = a * e - b * b
    s = np.where(den > 1e-12, np.clip((b * f - c * e) / den, 0, 1), 0.0)
    t = np.clip((b * s + f) / e, 0, 1)
    s = np.clip((b * t - c) / a, 0, 1)
    return np.linalg.norm((P1 + s[:, None] * d1) - (Q1 + t[:, None] * d2), axis=1)


def iso_dirs(n, rng):
    v = rng.normal(size=(n, 3)); return v / np.linalg.norm(v, axis=1, keepdims=True)


def oblate_dirs(n, rng, c=0.5):                      # in-plane preferred
    v = rng.normal(size=(n, 3)); v[:, 2] *= c
    return v / np.linalg.norm(v, axis=1, keepdims=True)


def board_count(n, rng, dirfun):
    L = rng.choice(ell, size=n, p=p); X = rng.uniform(0, 1, size=(n, 3))
    v = dirfun(n, rng)
    P1, P2 = X - 0.5 * L[:, None] * v, X + 0.5 * L[:, None] * v
    i, j = np.triu_indices(n, 1)
    return int(np.sum(seg_seg_dist(P1[i], P2[i], P1[j], P2[j]) <= twor))


def mean_sine(dirfun, M=1_000_000):
    a = dirfun(M, rng); b = dirfun(M, rng)
    return np.sqrt(1 - np.clip((a * b).sum(1), -1, 1) ** 2).mean()


reps = 2000
ns = [100, 200, 400, 800]
samples = {n: np.array([board_count(n, rng, iso_dirs) for _ in range(reps)], float) for n in ns}

# --- coverage + Kolmogorov distance ---
cov, dk = [], []
for n in ns:
    z = (samples[n] - samples[n].mean()) / samples[n].std(ddof=1)
    row = {"n": n}
    for lvl in (0.90, 0.95, 0.99):
        row[f"cov{int(lvl*100)}"] = np.mean(np.abs(z) <= stats.norm.ppf(0.5 + lvl / 2))
    row["upper95"] = np.mean(z <= stats.norm.ppf(0.95))
    cov.append(row)
    dk.append({"n": n, "d_K": stats.kstest(z, "norm").statistic,
               "mc_noise_floor": 0.8687 / np.sqrt(reps)})
pd.DataFrame(cov).to_csv(f"{DATA_DIR}/coverage.csv", index=False)
pd.DataFrame(dk).to_csv(f"{DATA_DIR}/ks_distance.csv", index=False)
print(pd.DataFrame(cov).to_string(index=False))

# --- calibration figure ---
def cov_curve(c):
    z = (c - c.mean()) / c.std(ddof=1); nom = np.linspace(0.50, 0.99, 25)
    return nom, np.array([np.mean(np.abs(z) <= stats.norm.ppf(0.5 + t / 2)) for t in nom])
fig, ax = plt.subplots(figsize=(4.4, 3.4))
ax.plot([0.5, 1], [0.5, 1], color="0.6", ls="--", lw=1, label="identity")
for n, col, mk in [(100, "0.45", "o"), (800, "black", "s")]:
    xx, yy = cov_curve(samples[n])
    ax.plot(xx, yy, color=col, lw=1.4, marker=mk, ms=4, label=f"n = {n}")
ax.set_xlabel("nominal coverage"); ax.set_ylabel("empirical coverage")
ax.legend(frameon=False, loc="upper left")
for ext in (".png", ".pdf"):
    fig.savefig(f"{FIG_DIR}/fig_coverage{ext}", dpi=600, bbox_inches="tight")
plt.show()

# --- anisotropy comparison (isotropic vs oblate c=0.5) at n=400, 800 ---
Es = {"isotropic": mean_sine(iso_dirs), "oblate_c0.5": mean_sine(oblate_dirs)}
arows = []
for name, dfun in [("isotropic", iso_dirs), ("oblate_c0.5", oblate_dirs)]:
    r = {"law": name, "E_sin_gamma": Es[name]}
    for n in (400, 800):
        c = np.array([board_count(n, rng, dfun) for _ in range(reps)], float)
        z = (c - c.mean()) / c.std(ddof=1)
        r[f"mean_n{n}"] = c.mean(); r[f"skew_n{n}"] = stats.skew(c)
        r[f"dK_n{n}"] = stats.kstest(z, "norm").statistic
    arows.append(r)
pd.DataFrame(arows).to_csv(f"{DATA_DIR}/anisotropy.csv", index=False)
print(pd.DataFrame(arows).to_string(index=False))

# --- computational cost (wall time per realization) ---
trow = []
for n in ns:
    t = time.time(); [board_count(n, rng, iso_dirs) for _ in range(30)]
    trow.append({"n": n, "ms_per_rep": (time.time() - t) / 30 * 1000})
tim = pd.DataFrame(trow); tim["ratio_to_prev"] = tim["ms_per_rep"] / tim["ms_per_rep"].shift(1)
tim.to_csv(f"{DATA_DIR}/timing.csv", index=False)
print(tim.to_string(index=False))
